# Phase 1 data: Tier-B top-up + evaluation set + preselection

**Repo:** `octoborg-x/remove-photo-bg` · **branch:** `feat/distill-phase1-data`
**Spec:** `optimize_model_plan/audit.md` §6-Phase1

This notebook does the remaining Phase-1 work and stops for review:

1. Clone the branch that has the fixed fetchers (`manifest_io.py`, `image_ops.py`,
   `commons_api.py`, `fetch-eval.py`, Tier-B categories, leakage guard).
2. Fetch **Tier-B** training images (hair / fur / soft edges) — closes the gap
   where museum sources produced no soft-edge rows.
3. Fetch the **evaluation set** (`data/eval-manifest.csv`, `split=test`) from
   licence-gated Commons categories disjoint from training.
4. Run **preselection** (`select.py`) with the eval-set leakage guard.
5. Print real per-source / tier / split / category counts.

No GPU is required (all HTTP + PIL + scipy). Every cell is resumable — the
fetchers merge into the manifest by sha256, so re-running never duplicates.

> **Never invent numbers.** Report exactly what the last cell prints.


## 1. Runtime check

In [ ]:
import os, sys, shutil, subprocess
print("Hostname:", os.uname().nodename)
print("Python:", sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "(no GPU - fine, fetching is CPU-only)"
print("Disk:", shutil.disk_usage("/content"))


## 2. Clone the branch (or update an existing clone)

In [ ]:
import os, subprocess

BRANCH = "feat/distill-phase1-data"
REPO_URL = "https://github.com/octoborg-x/remove-photo-bg.git"
REPO = "/content/remove-photo-bg"

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)


## 3. Dependencies

In [ ]:
%pip install -q pillow numpy scipy requests
import PIL, numpy, scipy, requests
print("deps ok:", PIL.__version__, numpy.__version__, scipy.__version__)


## 4. Tier-B training fetch (hair / fur / soft edges)

Appends to `data/manifest.csv`. Categories are CC0/PD-heavy and disjoint from the
evaluation categories, so `select.py` tiers them **B**.


In [ ]:
!cd /content/remove-photo-bg && python3 scripts/distill/fetch-cc0.py --categories hair,fur --count 120 2>&1 | tail -5


## 5. Evaluation set fetch

Writes `data/eval-manifest.csv` (`split=test`) and `data/eval-images/`. Never
trained on or tuned; every metric downstream is "agreement with teacher, not
accuracy" (no ground-truth masks).


In [ ]:
!cd /content/remove-photo-bg && python3 scripts/distill/fetch-eval.py --per-category 15 2>&1 | tail -5


## 6. Preselection (deterministic) with leakage guard

Rejects: images in the eval set, missing files, short side < 768, low sharpness,
near-duplicates. Writes `data/manifest-selected.csv` and
`data/manifest-rejected.csv`.


In [ ]:
!cd /content/remove-photo-bg && python3 scripts/distill/select.py \
    --manifest data/manifest.csv \
    --out data/manifest-selected.csv \
    --rejected-out data/manifest-rejected.csv 2>&1 | tail -20


## 7. Real counts (report these)

In [ ]:
import csv, json, os
from collections import Counter

def load(p):
    return list(csv.DictReader(open(p))) if os.path.exists(p) else []

raw = load("data/manifest.csv")
sel = load("data/manifest-selected.csv")
rej = load("data/manifest-rejected.csv")
ev  = load("data/eval-manifest.csv")

def grp(rows, key): return dict(Counter(r.get(key,"") for r in rows).most_common())
def reason(s):
    if s.startswith("short-side"): return "short-side"
    if s.startswith("sharpness"): return "sharpness"
    return s

summary = {
    "raw_rows": len(raw),
    "raw_by_source": grp(raw, "source"),
    "raw_by_category": grp(raw, "category"),
    "selected_rows": len(sel),
    "selected_by_split": grp(sel, "split"),
    "selected_by_tier": grp(sel, "tier"),
    "selected_by_category": grp(sel, "category"),
    "rejected_total": len(rej),
    "rejected_by_reason": dict(Counter(reason(r.get("reject_reason","")) for r in rej).most_common()),
    "eval_rows": len(ev),
    "eval_by_category": grp(ev, "category"),
}
print(json.dumps(summary, indent=2))
os.makedirs("smoke-results/distill", exist_ok=True)
json.dump(summary, open("smoke-results/distill/phase1-counts.json","w"), indent=2)
print("\nwrote smoke-results/distill/phase1-counts.json")


## 8. STOP

Phase 1 is complete when the counts above are real and the eval set is ≥ 100
images. Report the JSON from cell 7 back for review **before** any training
(spec §6-Phase1.4 STOP).

If a fetcher printed a low number, re-run its cell — it resumes.
